In [1]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, Dataset

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
# import seaborn as sns

from tqdm.notebook import trange, tqdm

import random

random.seed(42)


# plt.style.use('seaborn-v0_8-darkgrid')
# sns.set_palette('husl')
torch.manual_seed(42)

In [2]:
# Hyperparamters
learning_rate = 1e-4
# n_epochs = 20
n_epochs = 2 # just to get things setup
batch_size = 32


In [3]:
label_mapping = {
    "LTR": 0,
    "Non-LTR/LINE": 1,
    "SINE": 2,
    "TIR": 3,
    "Helitron": 4,
    "Crypton": 5,
    "DIRS": 6,
    "Maverick": 7,
    "Penelope-like Element/PLE": 8
}

sequence_encoding = {
    'A': [1, 0, 0, 0],
    'C': [0, 1, 0, 0],
    'G': [0, 0, 1, 0],
    'T': [0, 0, 0, 1]
}


In [4]:
""" Padding code from GLARE -> they used 4096 as their max_length """

# Random crop for training augmentation
# if self.augment and len(sequence) > self.max_length:
#     start = random.randint(0, len(sequence) - self.max_length)
#     sequence = sequence[start : start + self.max_length]

# # Reverse-complement augmentation (stochastic, training only).
# if self.rc_augment and random.random() < 0.5:
#     sequence = sequence.translate(RC_MAP)[::-1]

# encoding = self.tokenizer(
#     sequence,
#     return_tensors="pt",
#     padding="max_length",
#     truncation=True,
#     max_length=self.max_length,
#     pad_to_multiple_of=128,
# )

' Padding code from GLARE -> they used 4096 as their max_length '

In [23]:
# 4096 is what GLARE used.
# MAX_SEQ_LENGTH = 4096
# start with this smaller for testing because I have a feeling it is going to be slow
MAX_SEQ_LENGTH = 1000 


def custom_collate_fn(batch):
    """ Custom collate function for padding sequences up to the length of the longest sequence in the batch
    also returns the lengths of the sequences prior to padding """

    sequences, labels = zip(*batch)
    padded_sequences = pad_sequences(sequences, batch_first=True)
    lengths = torch.tensor([len(s) for s in sequences])
    
    return padded_sequences, lengths, labels

def one_hot(sequence):
    encoded = []
    for nucleotide in sequence:
        encoded.append(sequence_encoding.get(nucleotide, "UNKNOWN"))
    return torch.tensor(encoded, dtype=torch.float32)


class CSVDataset(Dataset):
    
    def __init__(self, csv_file, max_length=MAX_SEQ_LENGTH, train=False):

        df = pd.read_csv(csv_file)
        
        self.sequences = df["sequence_content"].values
        self.max_length = max_length
        self.train = train

        # for sequence in self.sequences:
        #     # print (len(sequence))
        #     for nucleotide in sequence:
        #         if nucleotide != 'A' and nucleotide != 'C' and nucleotide != 'G' and nucleotide != 'T':
        #             print (f"Problem with bad nucleotide {nucleotide}")

        print(f"Longest sequence: {df["sequence_content"].str.len().max()}")
        print(f"Shortest sequence: {df["sequence_content"].str.len().min()}")
        print(f"Mean sequence length: {df["sequence_content"].str.len().mean()}")
        
        self.encoded_labels = df["TE_Order"].map(label_mapping)
        assert self.encoded_labels.notna().all(), df.loc[encoded_labels.isna(), "TE_Order"].unique()
        
        # self.X = torch.tensor(encoded_sequences, dtype=torch.float32)
        self.y = torch.tensor(self.encoded_labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        sequence = self.sequences[idx]

        if len(sequence) > self.max_length:
            print(f"need to crop sequence of length {len(sequence)}")
            start = 0

            # using same train cropping technique as GLARE
            if self.train == True:
                start = random.randint(0, len(sequence) - self.max_length)
            else:
                # TODO: add test cropping technique
                print("Do the thing")
            
            sequence = sequence[start : start + self.max_length]
            

        return one_hot(sequence), self.y[idx]

training_data = CSVDataset(
    csv_file="LSTMDataSets/PREPROCESSED_TE_feature_extraction_dataset_without_maverickpolinton.csv", 
    max_length=MAX_SEQ_LENGTH,
    train=True
)
# testing_data = CSVDataset(csv_file="Preprocessed/test.csv")

for i in range(10):
     print(training_data[i])



Longest sequence: 39893
Shortest sequence: 43
Mean sequence length: 2942.278810237491
(tensor([[0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        ...,
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]]), tensor(2))
(tensor([[0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        ...,
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]]), tensor(2))
(tensor([[0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        ...,
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]]), tensor(2))
(tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [1., 0., 0., 0.],
        ...,
        [1., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 0., 1.]]), tensor(2))
(tensor([[0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        ...,
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]]), tensor(2))

In [24]:
data_loader_train = DataLoader(training_data, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)

In [7]:
# contructor defines the layers
# forward describes how the data flows through the layers


class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size
        )

        # classification layer (fully connected)
        self.fc_out = nn.Linear(hidden_size, output_size)
    
    def forward(self, input_seq):

        # output contains hidden state from each timestamp, from final lstm layer
        # hidden_out is the final hidden state for each LSTM layer -> use -1 to get last layer
        output, (hidden_out, mem_out) = self.lstm(input_seq)

        final_hidden = hidden_out[-1]
        
        return self.fc_out(final_hidden)
        
        


In [8]:
device = torch.device(0 if torch.cuda.is_available() else 'cpu') # update for macOS specific

In [9]:
input_size = 4
hidden_size = 128
output_size = 9 # 9 TE output classes

# Create model
lstm_classifier = LSTM(input_size, hidden_size, output_size)

# Initialize the optimizer with the above parameters
optimizer = optim.Adam(lstm_classifier.parameters(), lr=learning_rate)

# Define the loss function
loss_fn = nn.CrossEntropyLoss()


In [10]:
num_model_params = 0

for param in lstm_classifier.parameters():
    num_model_params += param.flatten().shape[0]

print(f"The model has {num_model_params} parameters")

The model has 69769 parameters


In [11]:
training_loss_logger = []
test_loss_logger = []

training_acc_logger = []
testing_acc_logger = []

In [13]:
train_acc = 0
test_acc = 0

for epoch in range(n_epochs):

    print (epoch)

    # set model to training mode
    lstm_classifier.train()
    steps = 0

    # Go through training data
    for labels, sequences in training_data:

        batch_size = label.shape[0]

        # do something like one_hot encoding for DNA sequences
        
        # Initialize hidden and memory states
        hidden = torch.zeros(num_layers, batch_size, hidden_size, device = device)
        memory = torch.zeros(num_layers, batch_size, hidden_size, device = device)

        # forward pass
        pred, hidden, memory = lstm_classifier(sequences)
    
        # loss calculation

        # backpropagation and optimization

        # append training loss to logger

        # calculate training accuracy


0


NameError: name 'label' is not defined